# 03 — Vanilla GRPO baseline

Trains the actor with the standard group-normalized advantage and **real tool calls**.
Used as the primary baseline for comparison against Dyna-GRPO.

**Hardware**: 1× A100-80G; 2× A100 if you increase `batch_size`.
**Time**: 4-8 hrs for 500 steps. Loss curves logged to `/workspace/dyna_grpo/logs/`.


In [ ]:
import sys, os; sys.path.insert(0, str(os.path.abspath(os.path.join(os.getcwd(), '..'))))
import json, time, copy, torch
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model
from dyna_grpo.config import MODEL, GRPO, PATHS
from dyna_grpo.data import aime_2024, numina_math_subset
from dyna_grpo.rewards import reward_for
from dyna_grpo.grpo import GRPOTrainer
from dyna_grpo.utils import set_seed, save_metrics, logger
from dyna_grpo.trace_collector import Trajectory
set_seed(GRPO.seed)

In [ ]:
tok = AutoTokenizer.from_pretrained(MODEL.actor_name, trust_remote_code=True)
if tok.pad_token is None: tok.pad_token = tok.eos_token

def load_actor():
    base = AutoModelForCausalLM.from_pretrained(
        MODEL.actor_name, torch_dtype=torch.bfloat16, device_map='cuda:0',
        trust_remote_code=True)
    cfg = LoraConfig(r=MODEL.lora_r, lora_alpha=MODEL.lora_alpha,
                     lora_dropout=MODEL.lora_dropout, bias='none',
                     target_modules=list(MODEL.lora_target_modules), task_type='CAUSAL_LM')
    return get_peft_model(base, cfg)

actor = load_actor()
ref = AutoModelForCausalLM.from_pretrained(
    MODEL.actor_name, torch_dtype=torch.bfloat16, device_map='cuda:1' if torch.cuda.device_count() > 1 else 'cuda:0',
    trust_remote_code=True).eval()
for p in ref.parameters(): p.requires_grad = False
actor.print_trainable_parameters()

In [ ]:
# Training prompts: math problems with known numeric answers
import random
from datasets import load_dataset

# Use NuminaMath-CoT for training; AIME-2024 held out for eval
train_pool = []
for r in numina_math_subset(2000):
    # Field already 'problem' after rename
    train_pool.append({'prompt': r['problem'], 'answer': None, 'kind': 'aime'})  # rule-based reward needs answer
# Filter for problems with extractable integer answers
import re
ANS_RE = re.compile(r'\\boxed\{(-?\d+)\}')
def with_gold(r):
    m = ANS_RE.search(r.get('problem', '') + ' ' + r.get('solution', ''))
    if m:
        try: return int(m.group(1))
        except: return None
    return None

raw = numina_math_subset(8000)
train_pool = []
for r in raw:
    g = with_gold(r)
    if g is not None and abs(g) < 10000:
        train_pool.append({'prompt': r['problem'], 'answer': g, 'kind': 'aime'})
    if len(train_pool) >= 500:
        break
print('Training prompts:', len(train_pool))

In [ ]:
def gen_fn(ctx, max_new):
    enc = tok(ctx, return_tensors='pt', truncation=True, max_length=6000).to('cuda:0')
    with torch.no_grad():
        o = actor.generate(**enc, max_new_tokens=max_new, do_sample=True,
                            temperature=1.0, top_p=0.9,
                            pad_token_id=tok.pad_token_id)
    return tok.decode(o[0][enc.input_ids.size(1):], skip_special_tokens=True)

def reward_fn(traj: Trajectory, sample: dict) -> float:
    return reward_for(sample['kind'], traj.flat_text(), sample)

opt = torch.optim.AdamW([p for p in actor.parameters() if p.requires_grad],
                          lr=GRPO.learning_rate)
trainer = GRPOTrainer(actor, ref, tok, opt, gen_fn, reward_fn, 'cuda:0', GRPO)

In [ ]:
history = []
log_path = Path(PATHS['logs']) / 'baseline_grpo.jsonl'
ckpt_dir = Path(PATHS['ckpts']) / 'baseline_grpo'
ckpt_dir.mkdir(parents=True, exist_ok=True)

import random
random.seed(GRPO.seed)
for step in range(GRPO.total_steps):
    batch = random.sample(train_pool, GRPO.batch_size)
    prompts_with_samples = [(b['prompt'], b) for b in batch]
    info = trainer.step(prompts_with_samples)
    info['step'] = step
    history.append(info)
    if step % GRPO.log_every == 0:
        print(f'step={step} reward={info["reward_mean"]:.3f} kl={info["kl"]:.4f} ratio={info["ratio_mean"]:.3f}')
    if step % GRPO.save_every == 0 and step > 0:
        actor.save_pretrained(ckpt_dir / f'step{step}')
    save_metrics(log_path, {'history': history})
actor.save_pretrained(ckpt_dir / 'final')
print('Baseline GRPO done.')